In [1]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

# Dataset klarifikasi Komdigi
DATA_PATH = Path("data/tabular/komdigi_hoaks.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("../../../data/tabular/komdigi_hoaks.csv")
if not DATA_PATH.exists():
    raise FileNotFoundError("File data/tabular/komdigi_hoaks.csv tidak ditemukan.")

dataset = pd.read_csv(DATA_PATH, usecols=["title", "body_text", "topics"])
dataset["text"] = dataset["title"].fillna("") + " " + dataset["body_text"].fillna("")
dataset["topics"] = dataset["topics"].fillna("").str.strip()
dataset = dataset.loc[dataset["text"].str.strip().ne("") & dataset["topics"].ne("")].copy()

label_encoder = LabelEncoder()
y = label_encoder.fit_transform(dataset["topics"])
X_train_text, X_test_text, y_train, y_test = train_test_split(
    dataset["text"], y, test_size=0.2, random_state=42, stratify=y
)

# TF-IDF hanya di-fit pada data latih untuk mencegah kebocoran data.
vectorizer = TfidfVectorizer(max_features=256, ngram_range=(1, 2), min_df=2)
X_train = vectorizer.fit_transform(X_train_text).toarray().astype(np.float32)
X_test = vectorizer.transform(X_test_text).toarray().astype(np.float32)
num_classes = len(label_encoder.classes_)
y_train_one_hot = np.eye(num_classes, dtype=np.float32)[y_train]
class_weights = len(y_train) / (num_classes * np.bincount(y_train))

print(f"Jumlah data: {len(dataset)}")
print(f"Fitur TF-IDF: {X_train.shape[1]}")
print("Kelas topik:", list(label_encoder.classes_))

# Fungsi aktivasi
def sigmoid(x):
    x = np.clip(x, -500, 500)
    return 1 / (1 + np.exp(-x))

def sigmoid_derivative(a):
    return a * (1 - a)

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return (x > 0).astype(float)

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

# ANN dengan output softmax untuk klasifikasi multi-kelas.
def train_ann(hidden_size, activation, epochs=100, lr=0.1, batch_size=64):
    rng = np.random.default_rng(42)
    input_size = X_train.shape[1]
    limit1 = np.sqrt(6 / (input_size + hidden_size))
    limit2 = np.sqrt(6 / (hidden_size + num_classes))
    W1 = rng.uniform(-limit1, limit1, (input_size, hidden_size)).astype(np.float32)
    b1 = np.zeros((1, hidden_size), dtype=np.float32)
    W2 = rng.uniform(-limit2, limit2, (hidden_size, num_classes)).astype(np.float32)
    b2 = np.zeros((1, num_classes), dtype=np.float32)

    for epoch in range(epochs):
        indices = rng.permutation(len(X_train))
        for start in range(0, len(indices), batch_size):
            batch_indices = indices[start:start + batch_size]
            X_batch = X_train[batch_indices]
            y_batch = y_train_one_hot[batch_indices]
            weights = class_weights[y_train[batch_indices]].astype(np.float32)

            z1 = X_batch @ W1 + b1
            a1 = relu(z1) if activation == "relu" else sigmoid(z1)
            probabilities = softmax(a1 @ W2 + b2)

            d_z2 = (probabilities - y_batch) * weights[:, None]
            d_z2 /= np.sum(weights)
            d_W2 = a1.T @ d_z2
            d_b2 = np.sum(d_z2, axis=0, keepdims=True)
            d_a1 = d_z2 @ W2.T
            d_z1 = d_a1 * relu_derivative(z1) if activation == "relu" else d_a1 * sigmoid_derivative(a1)
            d_W1 = X_batch.T @ d_z1
            d_b1 = np.sum(d_z1, axis=0, keepdims=True)

            W1 -= lr * d_W1
            b1 -= lr * d_b1
            W2 -= lr * d_W2
            b2 -= lr * d_b2

        if epoch % 50 == 0 or epoch == epochs - 1:
            z1 = X_train @ W1 + b1
            a1 = relu(z1) if activation == "relu" else sigmoid(z1)
            train_probabilities = softmax(a1 @ W2 + b2)
            loss = -np.mean(np.log(train_probabilities[np.arange(len(y_train)), y_train] + 1e-8))
            print(f"Epoch {epoch}, Loss: {loss:.6f}")

    z1 = X_test @ W1 + b1
    a1 = relu(z1) if activation == "relu" else sigmoid(z1)
    test_probabilities = softmax(a1 @ W2 + b2)
    predictions = np.argmax(test_probabilities, axis=1)
    loss = -np.mean(np.log(test_probabilities[np.arange(len(y_test)), y_test] + 1e-8))
    return loss, predictions


# Perbandingan jumlah neuron dan fungsi aktivasi
experiments = [
    ("8 Neuron + Sigmoid", 8, "sigmoid"),
    ("16 Neuron + Sigmoid", 16, "sigmoid"),
    ("16 Neuron + ReLU", 16, "relu"),
]
results = {}
for name, hidden_size, activation in experiments:
    print(f"\n=== {name} ===")
    loss, predictions = train_ann(hidden_size, activation)
    accuracy = accuracy_score(y_test, predictions)
    macro_f1 = f1_score(y_test, predictions, average="macro", zero_division=0)
    results[name] = (loss, accuracy, macro_f1, predictions)
    print(f"Test Loss: {loss:.6f} | Accuracy: {accuracy:.4f} | Macro-F1: {macro_f1:.4f}")

best_name = max(results, key=lambda name: results[name][2])
best_predictions = results[best_name][3]
print(f"\n=== Hasil terbaik berdasarkan Macro-F1: {best_name} ===")
print(classification_report(y_test, best_predictions, target_names=label_encoder.classes_, zero_division=0))
print("Confusion matrix:")
print(confusion_matrix(y_test, best_predictions))


Jumlah data: 4176
Fitur TF-IDF: 256
Kelas topik: ['Hoaks', 'Kesehatan', 'Masyarakat Digital', 'Pejabat Publik', 'Penipuan', 'Program/Kebijakan']

=== 8 Neuron + Sigmoid ===
Epoch 0, Loss: 1.766642
Epoch 50, Loss: 1.409047
Epoch 99, Loss: 1.252207
Test Loss: 1.332440 | Accuracy: 0.4199 | Macro-F1: 0.2730

=== 16 Neuron + Sigmoid ===
Epoch 0, Loss: 1.647012
Epoch 50, Loss: 1.497948
Epoch 99, Loss: 1.095752
Test Loss: 1.180597 | Accuracy: 0.5191 | Macro-F1: 0.2961

=== 16 Neuron + ReLU ===
Epoch 0, Loss: 1.671189
Epoch 50, Loss: 1.016337
Epoch 99, Loss: 0.633904
Test Loss: 0.937399 | Accuracy: 0.6376 | Macro-F1: 0.3349

=== Hasil terbaik berdasarkan Macro-F1: 16 Neuron + ReLU ===
                    precision    recall  f1-score   support

             Hoaks       0.93      0.65      0.77       708
         Kesehatan       0.14      0.27      0.18        11
Masyarakat Digital       0.05      0.15      0.08        20
    Pejabat Publik       0.24      0.63      0.35        27
          Pen